# 04 — Who is allowed to read this?

**Access pattern: authorisation, across clouds.** Notebook 03 showed that a public
bucket is just HTTPS. That is true, and it is the whole of it — for *some* buckets.

This one asks the question 03 skipped: **how do you know whether you are allowed?**
The answer is a status code, and the codes are not interchangeable.

Nothing here needs an account. Every request below is made anonymously, with no
credentials of any kind, and the point is watching what each service says.

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S

# A real requests.Session, with a disk cache. Everything you use below -- .get,
# .text, .content, .json, .status_code, .raise_for_status, params= -- is
# ordinary `requests`. The cache is invisible at the call site.
http = _fetch.session()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

_fetch.OFFLINE and print("offline mode: serving prefetched responses")

False

## 0. This notebook does not use the cache

Everything else in this workshop goes through `_fetch.session()`, which caches every
response to disk. This notebook deliberately uses **plain `requests`**, and the reason
is the subject matter.

A cache stores the answer you got. For this notebook the answer is *"what does this
service's access policy say right now"* — and access policy changes. A bucket that was
public last month can be made Requester Pays tomorrow, and a cached `403` would then be
sitting in your cache folder asserting, confidently and wrongly, that you may not read
it.

**A cache is only honest for facts that do not change.** Everything else you fetch
should be treated as a measurement with a timestamp, not as a fact. That is the one
piece of judgement this workshop cannot hand you in a function call.

Which also means this notebook needs a live network, and CI runs it in a job that has
one.

In [2]:
# Plain requests. No session, no cache, no _fetch.
import xml.etree.ElementTree as ET

import requests

S3_NS = "{http://s3.amazonaws.com/doc/2006-03-01/}"
# Note the asymmetry: a successful listing is <ListBucketResult xmlns="..."> with
# namespaced children, but a failure is a bare <Error> whose children are unprefixed.
# So findtext(f"{S3_NS}Code") returns None on *every* error -- a silent failure, and the
# reason this cell prints the code and the message side by side rather than trusting one.


def s3_listing(bucket, max_keys=5):
    """List a bucket anonymously. Returns the raw Response, deliberately unparsed."""
    return requests.get(
        f"https://{bucket}.s3.amazonaws.com/",
        params={"list-type": "2", "max-keys": max_keys},
        timeout=30,
    )


def error_code(resp):
    """Pull <Code> out of an S3 error body, or None if this was a success."""
    if resp.status_code == 200:
        return None
    try:
        return ET.fromstring(resp.content).findtext("Code")   # no namespace
    except ET.ParseError:
        return None


def error_message(resp):
    """Pull <Message> out of an S3 error body."""
    try:
        return (ET.fromstring(resp.content).findtext("Message") or "").strip()
    except ET.ParseError:
        return ""

## 1. Four buckets, one identical request

Same method, same URL shape, same absence of credentials. Four different answers.

`noaa-goes16` is a real NOAA bucket of GOES-16 satellite imagery, and it is open.
`landsat-pds` and `usgs-landsat` are both real, well-known USGS Landsat buckets, and
neither will serve you anonymously. `noaa-pds` does not exist at all — I guessed it.

In [3]:
BUCKETS = [
    ("noaa-goes16",  "a real, open NOAA bucket"),
    ("landsat-pds",  "real, and well known"),
    ("usgs-landsat", "also real, also well known"),
    ("noaa-pds",     "a name I invented"),
]

print(f"  {'bucket':<14}{'status':<8}{'S3 code':<15}message")
print("  " + "-" * 118)
for bucket, note in BUCKETS:
    r = s3_listing(bucket)
    msg = error_message(r) or "(no error -- this was a success)"
    # Not truncated. The whole point of the fourth row is that this sentence names the
    # policy, and a message cut off at "Requester" stops doing that job.
    print(f"  {bucket:<14}{r.status_code:<8}{str(error_code(r)):<15}{msg}")
    print(f"  {'':<14}{'':<8}{'':<15}({note})")

  bucket        status  S3 code        message
  ----------------------------------------------------------------------------------------------------------------------


  noaa-goes16   200     None           (no error -- this was a success)
                                       (a real, open NOAA bucket)


  landsat-pds   403     AccessDenied   Access Denied
                                       (real, and well known)


  usgs-landsat  403     AccessDenied   Anonymous users cannot invoke requests against Requester Pays buckets. Please authenticate.
                                       (also real, also well known)


  noaa-pds      404     NoSuchBucket   The specified bucket does not exist
                                       (a name I invented)


### Read those as four different problems

`200` — you may read this bucket, anonymously, with nothing to sign and nothing to pay.

`404` `NoSuchBucket` — the bucket does not exist and never did. **No credential will make
it appear.** Your request was fine; your *name* was wrong.

`403` `AccessDenied` — the bucket is real and you are not on its list. Now, and only
now, credentials are the answer.

So `403` and `404` are not the same failure, even though `assert resp.ok` collapses them
into one. That is an afternoon lost: you go looking for an API key to fix a typo.
**Print the error code before you go looking for a key.** Same species of bug as
notebook 07’s `99`-versus-`999` sentinels — a failure whose cause is not the one you would
guess from the symptom.

### The part that is genuinely not obvious

Look at the two `403` rows again. **They do not say the same thing.**

- `usgs-landsat` names the problem outright: *"Anonymous users cannot invoke requests
  against Requester Pays buckets. Please authenticate."* That message gives you the
  policy and the fix.
- `landsat-pds` says only *"Access Denied"*. Which is true, and close to useless — it
  does not distinguish "you are not permitted" from "this costs money", and those have
  completely different answers.

So read the error body, but **do not expect it to name the policy.** Where it does not,
the registry entry is the source. That is trap-table rule 4 in one line: the thing the
service does not tell you is the thing you have to go and look up.

## 2. Requester Pays: public to look at, not free to fetch

`landsat-pds` is not secret. You can read its name in a registry, and anybody can see
that the data is open. What is not open is the **bandwidth**.

The bucket's owner pays for egress. A large bucket of multi-gigabyte scenes would be
ruinous to serve for free, so AWS splits the difference: the bucket is visible to
everyone and billable to whoever requests the bytes. Hence the name.

So the honest statement about a Requester Pays bucket is not "it is locked" but
**"it is public, and using it costs money"** — which is a different thing, and much
easier to get wrong, because nothing about the URL tells you which you are looking at.

In [4]:
# Both of these are real, well-known public datasets. One you may fetch anonymously;
# the other you may not. Nothing in the bucket name reveals which.
CASES = [
    ("noaa-goes16",  "GOES-16 satellite imagery", "free to fetch anonymously", ""),
    ("landsat-pds",  "Landsat scenes",            "Requester Pays",           "per the registry"),
    ("usgs-landsat", "Landsat scenes, other copy", "Requester Pays",          "the body says so"),
    ("commoncrawl",  "web crawl archives",        "Requester Pays",           "per the registry"),
]
print(f"  {'bucket':<14}{'status':<8}{'what it is':<30}{'verdict':<18}how we know")
print("  " + "-" * 100)
for bucket, what, verdict, how in CASES:
    r = s3_listing(bucket, max_keys=1)
    print(f"  {bucket:<14}{r.status_code:<8}{what:<30}{verdict:<18}{how}")
print()
print("  All four are public data. Only one is free to read.")
print("  The URL never told you which, and neither did two of the error bodies.")

  bucket        status  what it is                    verdict           how we know
  ----------------------------------------------------------------------------------------------------


  noaa-goes16   200     GOES-16 satellite imagery     free to fetch anonymously


  landsat-pds   403     Landsat scenes                Requester Pays    per the registry
  usgs-landsat  403     Landsat scenes, other copy    Requester Pays    the body says so


  commoncrawl   403     web crawl archives            Requester Pays    per the registry

  All four are public data. Only one is free to read.
  The URL never told you which, and neither did two of the error bodies.


### Free to access is not free to use

The distinction that catches people, and it is worth being precise about:

| | the bytes | running a query over them |
|---|---|---|
| public S3 bucket | free | **not free** — Athena and Redshift Spectrum bill per query |
| BigQuery public dataset | free | metered — 1 TiB/month free, then not |
| public GCS bucket (notebook 03) | free | **not free** — BigQuery bills the scan |

The data being open and the *computing* being free are independent facts, and the
second is the one that produces an invoice. An open bucket queried through Athena costs
money; the same bytes fetched with `requests` cost nothing. Identical data, opposite
bills, decided entirely by which tool you reached for.

## 3. The counter-example: BigQuery

Same phrase — "free public datasets" — and the opposite access model.

This is worth seeing precisely because the marketing language is identical to S3's.
BigQuery genuinely does host large public datasets, queryable with SQL, and the first
terabyte of scanning per month is free. It is also true that **you cannot make the
request at all** without authenticating.

In [5]:
BQ = ("https://bigquery.googleapis.com/bigquery/v2/projects/"
      "bigquery-public-data/queries")

# The cheapest possible query, against the public project, with no credentials.
r = requests.post(
    BQ,
    json={"query": "SELECT 1 AS ok", "useLegacySql": False},
    timeout=30,
)
print(f"  POST {BQ}")
print(f"  -> HTTP {r.status_code}")
body = r.json()
print()
print("  ", body["error"]["message"])

  POST https://bigquery.googleapis.com/bigquery/v2/projects/bigquery-public-data/queries
  -> HTTP 401

   Request is missing required authentication credential. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.


In [6]:
# Even *listing* the public datasets needs a principal.
r = requests.get(
    "https://bigquery.googleapis.com/bigquery/v2/projects/bigquery-public-data/datasets",
    timeout=30,
)
print(f"  listing datasets anonymously -> HTTP {r.status_code}")
print("  reason:", r.json()["error"]["errors"][0]["reason"])

  listing datasets anonymously -> HTTP 401
  reason: required


Read that against the S3 results. The data is public, the price is zero-or-free, and
the API still refuses to answer — because **authentication and authorisation are
different questions**, and this service requires an answer to the first one before it
will even consider the second.

| | anonymous | what you need |
|---|---|---|
| S3, public bucket | **works** | nothing |
| S3, Requester Pays | 403 | an account, and you pay egress |
| BigQuery, public dataset | **401** | a Google account, a project, and a token |

### The advice is out of date, which is why people disagree

The honest answer here has *changed*, and if your sources disagree, that is why:

- **For years the answer was** "you need a billing account attached to your project."
  That was true, and it is the advice still circulating.
- **Now there is a sandbox.** From Google's own documentation: the BigQuery sandbox
  "lets you experience BigQuery without providing a credit card or creating a billing
  account for your project." Create a project, query the public datasets, pay nothing.
- **Or** with billing enabled, the first **1 TiB of query data processed per month is
  free**.

So "it's totally free" and "you need a credit card" were both correct answers to this
question, at different times. A field moves, and confident second-hand answers are the
first casualty.

Which is the transferable lesson, and the reason this notebook ends where it does:
**when a service's answer depends on its commercial policy, the documentation is the
source and your memory is not.** Fetch the docs, and check the date on them.

## 4. What to do with this

Not memorise a list of codes. Learn the order of operations:

1. **Read the status code.** `200`, `403`, `404`, `401` are four different problems.
2. **Read the body when it is an error.** The body names the cause, and the cause
   decides the fix. A 404 will never be fixed with a key.
3. **Distinguish "you are not allowed" from "I do not know who you are".** 403 and 401
   are not interchangeable: 401 means *authenticate*, 403 means *then ask for more*.
4. **Ask whether it is free before assuming it is.** And if it is free to read, check
   separately whether it is free to *compute over*.
5. **Check the date on the documentation** when the answer involves pricing.

A quick reference, and the only part worth memorising:

| you sent | you got | it means | do this |
|---|---|---|---|
| a request, no credentials | **200** | allowed | proceed |
| a request, no credentials | **401** | who are you? | authenticate |
| a request, no credentials | **403** | known, still not allowed | read the body: policy, or requester-pays? |
| a request, no credentials | **404** | no such thing | fix the name — **not** the key |
| a request, credentials | **429** | too many requests | slow down and retry with backoff |